# 03. Обучение и сравнение генеративных моделей

Этот ноутбук объединяет в себе несколько этапов для удобства работы в Colab.


In [ ]:
# === Colab setup (skip if running locally) ===
import os, sys

IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    # Клонируем репо (первый раз) и переходим в него
    if not os.path.exists("/content/gan-reviews"):
        !git clone https://github.com/SultanKhassenov/gan-reviews.git /content/gan-reviews
    %cd /content/gan-reviews
    !pip install -q -r requirements-trainer.txt

# Добавляем корень репо в sys.path, чтобы импорты работали из notebooks/
sys.path.insert(0, os.path.abspath(".."))

# Mount Google Drive для сохранения чекпоинтов (опционально)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/gan-reviews/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "../checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)


## --- 03_train_wgan_gp ---


# 03. Обучение WGAN-GP на эмбеддингах

Conditional WGAN-GP. Условие — категория товара (10 классов).

**Что делаем:**
1. Загружаем (N, 768) эмбеддинги + (N,) labels
2. DataLoader
3. Тренируем 200 эпох
4. Сохраняем чекпоинт + loss curves
5. Генерируем сэмплы для оценки в notebook 05


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path

from models.wgan_gp import Generator, Critic, train_wgan_gp, sample

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

EMB_DIR = Path('data/embeddings')
X = np.load(EMB_DIR / 'X_cls.npy').astype('float32')
y = np.load(EMB_DIR / 'labels.npy')
print('X:', X.shape, 'y:', y.shape, 'classes:', y.max() + 1)


In [ ]:
# Стандартизация (важно для GAN — иначе шум сильно отличается от данных)
mean = X.mean(axis=0, keepdims=True)
std = X.std(axis=0, keepdims=True) + 1e-6
X_norm = (X - mean) / std
np.save(EMB_DIR / 'X_cls_mean.npy', mean)
np.save(EMB_DIR / 'X_cls_std.npy', std)

ds = TensorDataset(torch.from_numpy(X_norm), torch.from_numpy(y))
dl = DataLoader(ds, batch_size=64, shuffle=True, drop_last=True)


In [ ]:
NUM_CLASSES = int(y.max() + 1)
gen = Generator(z_dim=128, emb_dim=768, num_classes=NUM_CLASSES)
crit = Critic(emb_dim=768, num_classes=NUM_CLASSES)

history = train_wgan_gp(
    gen, crit, dl,
    n_epochs=200,
    n_critic=5,
    lambda_gp=10.0,
    lr=1e-4,
    device=device,
    log_every=10,
)


In [ ]:
torch.save({
    'gen': gen.state_dict(),
    'crit': crit.state_dict(),
    'history': history,
    'num_classes': NUM_CLASSES,
}, f'{CHECKPOINT_DIR}/wgan_gp.pth')
print('saved')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['d_loss'], label='D (critic)')
axes[0].plot(history['g_loss'], label='G')
axes[0].legend(); axes[0].set_title('Losses'); axes[0].set_xlabel('epoch')
axes[1].plot(history['gp']); axes[1].set_title('Gradient Penalty')
plt.tight_layout()
plt.savefig('results/wgan_gp_losses.png', dpi=120)
plt.show()


In [ ]:
# Генерируем сэмплы для оценки
fake, fake_labels = sample(gen, n_per_class=500, num_classes=NUM_CLASSES, device=device)
fake_np = fake.cpu().numpy()
# Обратная стандартизация
fake_denorm = fake_np * std + mean
np.save('data/embeddings/X_gen_wgan.npy', fake_denorm)
np.save('data/embeddings/y_gen_wgan.npy', fake_labels.cpu().numpy())
print('saved fake samples:', fake_denorm.shape)


## --- 04_train_vae ---


# 04. Обучение Conditional VAE на эмбеддингах

Бейзлайн для сравнения с WGAN-GP. Та же задача, другая семья моделей.

Требование курса: минимум 2 генеративные модели.


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path

from models.vae import CVAE, train_vae, sample_vae

device = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_DIR = Path('data/embeddings')
X = np.load(EMB_DIR / 'X_cls.npy').astype('float32')
y = np.load(EMB_DIR / 'labels.npy')
mean = np.load(EMB_DIR / 'X_cls_mean.npy')
std = np.load(EMB_DIR / 'X_cls_std.npy')
X_norm = (X - mean) / std

ds = TensorDataset(torch.from_numpy(X_norm), torch.from_numpy(y))
dl = DataLoader(ds, batch_size=64, shuffle=True, drop_last=True)


In [ ]:
NUM_CLASSES = int(y.max() + 1)
vae = CVAE(emb_dim=768, latent_dim=64, num_classes=NUM_CLASSES)

history = train_vae(vae, dl, n_epochs=100, lr=1e-3, beta=1.0,
                    device=device, log_every=10)


In [ ]:
torch.save({
    'vae': vae.state_dict(),
    'history': history,
    'num_classes': NUM_CLASSES,
}, f'{CHECKPOINT_DIR}/vae.pth')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['loss']); ax[0].set_title('Total loss')
ax[1].plot(history['recon'], label='recon')
ax[1].plot(history['kl'], label='kl')
ax[1].legend(); ax[1].set_title('Recon vs KL')
plt.savefig('results/vae_losses.png', dpi=120)
plt.show()


In [ ]:
fake, fake_labels = sample_vae(vae, n_per_class=500,
                                num_classes=NUM_CLASSES, device=device)
fake_denorm = fake.cpu().numpy() * std + mean
np.save('data/embeddings/X_gen_vae.npy', fake_denorm)
np.save('data/embeddings/y_gen_vae.npy', fake_labels.cpu().numpy())
print('saved', fake_denorm.shape)


## --- 05_compare_models ---


# 05. Сравнение моделей: WGAN-GP vs VAE vs реальные данные

**Метрики:**
- MMD (Maximum Mean Discrepancy) — общая и по классам
- Frechet Distance — общая и по классам

**Визуализация:**
- t-SNE: реальные vs сгенерированные
- UMAP: то же, другой алгоритм
- Per-class scatter — где модель работает лучше/хуже


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from metrics.mmd import compute_mmd, compute_mmd_per_class
from metrics.frechet import compute_frechet_distance, compute_frechet_per_class

EMB = Path('data/embeddings')
real = np.load(EMB / 'X_cls.npy').astype('float32')
real_y = np.load(EMB / 'labels.npy')
wgan = np.load(EMB / 'X_gen_wgan.npy').astype('float32')
wgan_y = np.load(EMB / 'y_gen_wgan.npy')
vae = np.load(EMB / 'X_gen_vae.npy').astype('float32')
vae_y = np.load(EMB / 'y_gen_vae.npy')

NUM_CLASSES = int(real_y.max() + 1)


## Метрики


In [ ]:
import pandas as pd

results = []
for name, X, y in [('WGAN-GP', wgan, wgan_y), ('VAE', vae, vae_y)]:
    mmd = compute_mmd(real, X)
    fd = compute_frechet_distance(real, X)
    results.append({'model': name, 'MMD': mmd, 'Frechet': fd})

df_metrics = pd.DataFrame(results)
print(df_metrics)
df_metrics.to_csv('results/metrics_overall.csv', index=False)


In [ ]:
# Метрики по классам
rows = []
for name, X, y in [('WGAN-GP', wgan, wgan_y), ('VAE', vae, vae_y)]:
    mmd_per = compute_mmd_per_class(real, X, real_y, y, NUM_CLASSES)
    fd_per = compute_frechet_per_class(real, X, real_y, y, NUM_CLASSES)
    for c in range(NUM_CLASSES):
        rows.append({
            'model': name, 'class': c,
            'MMD': mmd_per[c], 'Frechet': fd_per[c],
        })
df_per = pd.DataFrame(rows)
print(df_per)
df_per.to_csv('results/metrics_per_class.csv', index=False)


## t-SNE визуализация


In [ ]:
from sklearn.manifold import TSNE

# Объединяем для совместной проекции
n_sample = 1500
idx_r = np.random.choice(len(real), min(n_sample, len(real)), replace=False)
idx_w = np.random.choice(len(wgan), min(n_sample, len(wgan)), replace=False)
idx_v = np.random.choice(len(vae), min(n_sample, len(vae)), replace=False)

X_all = np.vstack([real[idx_r], wgan[idx_w], vae[idx_v]])
src = (['real']*len(idx_r) + ['wgan']*len(idx_w) + ['vae']*len(idx_v))

tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
Z = tsne.fit_transform(X_all)

fig, ax = plt.subplots(figsize=(9, 7))
for s, c in zip(['real', 'wgan', 'vae'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
    m = [i for i, v in enumerate(src) if v == s]
    ax.scatter(Z[m, 0], Z[m, 1], c=c, label=s, alpha=0.5, s=10)
ax.legend(); ax.set_title('t-SNE: real vs generated')
plt.savefig('results/tsne_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
